# 逻辑回归（多分类 Softmax 线性层）

把 `28×28` 像素拉平成 `784` 维，一层线性映射到 `10` 类。  
训练：小批量梯度下降 + 交叉熵。


## 1. 导入与随机种子


In [1]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.utils.data as Data
import torchvision

torch.manual_seed(1)


## 2. 超参数

可按机器情况改 `EPOCH`、`BATCH_SIZE`。


In [2]:
EPOCH = 1
BATCH_SIZE = 50
LR = 0.001

TEST_N = 2000


## 3. 数据集与 DataLoader


In [3]:
import numpy as np
from pathlib import Path

import torchvision
from mnist_from_raw import MNISTNumpyDataset, load_all_numpy, raw_files_available

if raw_files_available():
    train_x, train_y, te_imgs, te_lbls = load_all_numpy()
    train_data = MNISTNumpyDataset(train_x, train_y)
    train_loader = Data.DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_x = torch.from_numpy(np.ascontiguousarray(te_imgs[:TEST_N])).unsqueeze(1).float().div_(255.0)
    test_y = torch.from_numpy(te_lbls[:TEST_N].copy()).long()
    print("数据来源: data/raw")
else:
    MNIST_ROOT = Path("./mnist")
    download = not MNIST_ROOT.is_dir() or not any(MNIST_ROOT.iterdir())
    train_data = torchvision.datasets.MNIST(
        root=str(MNIST_ROOT),
        train=True,
        transform=torchvision.transforms.ToTensor(),
        download=download,
    )
    train_loader = Data.DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_data = torchvision.datasets.MNIST(
        root=str(MNIST_ROOT), train=False, download=download
    )
    test_x = torch.unsqueeze(test_data.test_data, dim=1).type(torch.FloatTensor)[:TEST_N] / 255.0
    test_y = test_data.test_labels[:TEST_N]
    print("数据来源: torchvision ->", MNIST_ROOT.resolve())
print("train batches (approx):", len(train_loader))
print("test_x:", test_x.shape, "test_y:", test_y.shape)


数据来源: data/raw
train batches (approx): 1200
test_x: torch.Size([2000, 1, 28, 28]) test_y: torch.Size([2000])


/tmp/ipykernel_3273475/156012612.py:11: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  test_x = torch.from_numpy(np.ascontiguousarray(te_imgs[:TEST_N])).unsqueeze(1).float().div_(255.0)


## 4. 模型：784 → 10


In [4]:
class LogisticNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(28 * 28, 10)

    def forward(self, x):
        logits = self.fc(x)
        return logits


model = LogisticNet()
print(model)


LogisticNet(
  (fc): Linear(in_features=784, out_features=10, bias=True)
)


## 5. 损失函数与优化器


In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()


## 6. 训练循环

每步：`view` 拉平 → 前向 → 损失 → `zero_grad` → `backward` → `step`。


In [6]:
def accuracy(logits, y):
    pred = logits.argmax(dim=1)
    return (pred == y).float().mean().item()

for epoch in range(EPOCH):
    for step, (b_x, b_y) in enumerate(train_loader):
        b_x = b_x.view(-1, 28 * 28)
        logits = model(b_x)
        loss = loss_fn(logits, b_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 50 == 0:
            with torch.no_grad():
                te_logits = model(test_x.view(-1, 28 * 28))
                acc = accuracy(te_logits, test_y)
            print(
                f"epoch={epoch} step={step} loss={loss.item():.4f} test_acc={acc:.4f}"
            )


/data1/zdguo/document-parsing/alextools/experiment/MNIST/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:869: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


epoch=0 step=0 loss=2.2988 test_acc=0.2185
epoch=0 step=50 loss=1.2220 test_acc=0.7430
epoch=0 step=100 loss=0.7212 test_acc=0.7775
epoch=0 step=150 loss=0.5902 test_acc=0.8290
epoch=0 step=200 loss=0.6839 test_acc=0.8430
epoch=0 step=250 loss=0.4998 test_acc=0.8505
epoch=0 step=300 loss=0.4431 test_acc=0.8530
epoch=0 step=350 loss=0.4185 test_acc=0.8580
epoch=0 step=400 loss=0.4484 test_acc=0.8580
epoch=0 step=450 loss=0.5916 test_acc=0.8605
epoch=0 step=500 loss=0.2719 test_acc=0.8620
epoch=0 step=550 loss=0.3988 test_acc=0.8680
epoch=0 step=600 loss=0.2680 test_acc=0.8715
epoch=0 step=650 loss=0.4699 test_acc=0.8705
epoch=0 step=700 loss=0.3262 test_acc=0.8775
epoch=0 step=750 loss=0.2869 test_acc=0.8725
epoch=0 step=800 loss=0.4406 test_acc=0.8765
epoch=0 step=850 loss=0.3488 test_acc=0.8760
epoch=0 step=900 loss=0.2021 test_acc=0.8765
epoch=0 step=950 loss=0.5004 test_acc=0.8790
epoch=0 step=1000 loss=0.5216 test_acc=0.8810
epoch=0 step=1050 loss=0.5179 test_acc=0.8815
epoch=0 ste

## 7. 看前 10 个测试预测


In [7]:
model.eval()
with torch.no_grad():
    out = model(test_x[:10].view(-1, 28 * 28))
    pred = out.argmax(dim=1).cpu().numpy()
print("pred:", pred)
print("true:", test_y[:10].numpy())


pred: [7 2 1 0 4 1 4 9 6 9]
true: [7 2 1 0 4 1 4 9 5 9]
